In [ ]:
#!/usr/bin/env python3
"""
shidoku.py — Generate (and optionally solve) Shidoku puzzles (4x4 Sudoku).

Rules:
- Grid is 4x4
- Digits are 1..4
- Each row/col contains 1..4 exactly once
- Each 2x2 box contains 1..4 exactly once

This file provides:
- generate_solution(seed=None): returns a completed valid 4x4 grid
- generate_puzzle(seed=None, clues=8, symmetry=False): returns a puzzle with a UNIQUE solution
- solve(grid): returns one solution (or None if unsatisfiable)
- count_solutions(grid, limit=2): counts solutions up to 'limit' (used to enforce uniqueness)
- pretty_print(grid): prints a grid nicely

Usage:
    python shidoku.py --clues 8 --seed 123
    python shidoku.py --solve "1..4....2..3...."   (16 chars: 1-4 or . / 0)

Notes:
- "clues" is the number of givens left in the puzzle (0..16). Typical: 6–10.
- Uniqueness is enforced by counting solutions (early exit after 2).
"""

from __future__ import annotations

import argparse
import random
from typing import List, Optional, Tuple

N = 4
DIGITS = [1, 2, 3, 4]
BOX = 2  # 2x2 boxes


Grid = List[List[int]]  # 0 = empty


def parse_grid(s: str) -> Grid:
    s = s.strip()
    if len(s) != 16:
        raise ValueError("Grid string must be exactly 16 characters (row-major).")
    vals: List[int] = []
    for ch in s:
        if ch in ".0":
            vals.append(0)
        elif ch in "1234":
            vals.append(int(ch))
        else:
            raise ValueError(f"Invalid character in grid: {ch!r}")
    return [vals[i * N : (i + 1) * N] for i in range(N)]


def grid_to_string(g: Grid) -> str:
    out = []
    for r in range(N):
        for c in range(N):
            out.append("." if g[r][c] == 0 else str(g[r][c]))
    return "".join(out)


def pretty_print(g: Grid) -> None:
    sep = "+-----+-----+"
    print(sep)
    for r in range(N):
        row = []
        for c in range(N):
            v = g[r][c]
            row.append("." if v == 0 else str(v))
        print(f"| {' '.join(row[:2])} | {' '.join(row[2:])} |")
        if r % BOX == BOX - 1:
            print(sep)


def is_valid_move(g: Grid, r: int, c: int, v: int) -> bool:
    # row/col
    for k in range(N):
        if g[r][k] == v:
            return False
        if g[k][c] == v:
            return False
    # box
    br = (r // BOX) * BOX
    bc = (c // BOX) * BOX
    for rr in range(br, br + BOX):
        for cc in range(bc, bc + BOX):
            if g[rr][cc] == v:
                return False
    return True


def find_empty(g: Grid) -> Optional[Tuple[int, int]]:
    for r in range(N):
        for c in range(N):
            if g[r][c] == 0:
                return (r, c)
    return None


def candidates(g: Grid, r: int, c: int) -> List[int]:
    return [v for v in DIGITS if is_valid_move(g, r, c, v)]


def _choose_mrv_cell(g: Grid) -> Optional[Tuple[int, int, List[int]]]:
    """
    Minimum Remaining Values (MRV) heuristic: pick the empty cell
    with the fewest legal candidates.
    """
    best: Optional[Tuple[int, int, List[int]]] = None
    for r in range(N):
        for c in range(N):
            if g[r][c] != 0:
                continue
            cand = candidates(g, r, c)
            if not cand:
                return (r, c, [])
            if best is None or len(cand) < len(best[2]):
                best = (r, c, cand)
                if len(cand) == 1:
                    return best
    return best


def solve(g: Grid, rng: Optional[random.Random] = None) -> Optional[Grid]:
    """
    Returns one solution (deep-copied) or None if unsatisfiable.
    If rng is provided, candidate order is randomized (useful for generating varied solutions).
    """
    # work on a copy
    grid = [row[:] for row in g]

    def backtrack() -> bool:
        cell = _choose_mrv_cell(grid)
        if cell is None:
            return True  # solved
        r, c, cand = cell
        if not cand:
            return False

        if rng is not None:
            rng.shuffle(cand)

        for v in cand:
            grid[r][c] = v
            if backtrack():
                return True
            grid[r][c] = 0
        return False

    return grid if backtrack() else None


def count_solutions(g: Grid, limit: int = 2) -> int:
    """
    Count solutions up to 'limit'. This is used for uniqueness checks:
    - return 0: no solution
    - return 1: unique
    - return >=2: multiple solutions (we stop early)
    """
    grid = [row[:] for row in g]
    count = 0

    def backtrack() -> None:
        nonlocal count
        if count >= limit:
            return

        cell = _choose_mrv_cell(grid)
        if cell is None:
            count += 1
            return

        r, c, cand = cell
        if not cand:
            return

        # deterministic order is fine for counting
        for v in cand:
            grid[r][c] = v
            backtrack()
            grid[r][c] = 0
            if count >= limit:
                return

    backtrack()
    return count


def generate_solution(seed: Optional[int] = None) -> Grid:
    """
    Generate a full valid 4x4 solution grid by solving an empty grid
    with randomized choices.
    """
    rng = random.Random(seed)
    empty = [[0] * N for _ in range(N)]
    sol = solve(empty, rng=rng)
    if sol is None:
        # This should never happen in 4x4, but keep it safe.
        raise RuntimeError("Failed to generate a solution grid.")
    return sol


def _sym_pair_index(r: int, c: int) -> Tuple[int, int]:
    """180-degree rotational symmetry partner."""
    return (N - 1 - r, N - 1 - c)


def generate_puzzle(
    seed: Optional[int] = None,
    clues: int = 8,
    symmetry: bool = False,
    max_attempts: int = 5000,
) -> Grid:
    """
    Generate a Shidoku puzzle with a UNIQUE solution.

    Args:
        seed: RNG seed for reproducibility.
        clues: number of given cells to keep (0..16). Typical: 6–10.
        symmetry: if True, removals are paired with 180° rotational symmetry.
        max_attempts: safety cap on removal attempts.

    Returns:
        puzzle grid (0 = empty)

    Raises:
        ValueError if clues out of range.
        RuntimeError if puzzle can't be produced within attempt cap.
    """
    if not (0 <= clues <= 16):
        raise ValueError("clues must be in [0, 16].")

    rng = random.Random(seed)
    sol = generate_solution(seed=seed)
    puzzle = [row[:] for row in sol]

    # Build a list of positions noting symmetry (we'll attempt to remove from these).
    positions = [(r, c) for r in range(N) for c in range(N)]
    rng.shuffle(positions)

    def current_clues() -> int:
        return sum(1 for r in range(N) for c in range(N) if puzzle[r][c] != 0)

    attempts = 0
    idx = 0

    while current_clues() > clues and attempts < max_attempts:
        if idx >= len(positions):
            rng.shuffle(positions)
            idx = 0

        r, c = positions[idx]
        idx += 1

        if puzzle[r][c] == 0:
            continue

        # Determine which cells to remove this step
        to_clear = [(r, c)]
        if symmetry:
            r2, c2 = _sym_pair_index(r, c)
            if (r2, c2) != (r, c):
                to_clear.append((r2, c2))

        # If removing a pair would drop below target clues, skip
        nonzero_in_set = sum(1 for (rr, cc) in to_clear if puzzle[rr][cc] != 0)
        if current_clues() - nonzero_in_set < clues:
            continue

        # Try removal
        saved = [(rr, cc, puzzle[rr][cc]) for (rr, cc) in to_clear]
        for rr, cc, _v in saved:
            puzzle[rr][cc] = 0

        # Check uniqueness
        nsol = count_solutions(puzzle, limit=2)
        if nsol != 1:
            # revert
            for rr, cc, v in saved:
                puzzle[rr][cc] = v

        attempts += 1

    if current_clues() != clues:
        raise RuntimeError(
            f"Failed to reach exactly {clues} clues with unique solution "
            f"(ended at {current_clues()} clues after {attempts} attempts). "
            f"Try a different seed, increase max_attempts, or disable symmetry."
        )

    return puzzle


def main() -> None:
    ap = argparse.ArgumentParser(description="Generate or solve Shidoku (4x4 Sudoku).")
    ap.add_argument("--seed", type=int, default=None, help="RNG seed (int).")
    ap.add_argument("--clues", type=int, default=8, help="Number of given cells to keep (0..16).")
    ap.add_argument("--symmetry", action="store_true", help="Enforce 180° rotational symmetry in removals.")
    ap.add_argument("--solve", type=str, default=None, help="Solve a given grid string (16 chars; 1-4 or . / 0).")
    ap.add_argument("--print-solution", action="store_true", help="Also print the solution for generated puzzles.")
    args = ap.parse_args()

    if args.solve is not None:
        g = parse_grid(args.solve)
        print("Input:")
        pretty_print(g)
        sol = solve(g)
        if sol is None:
            print("No solution.")
            return
        print("Solution:")
        pretty_print(sol)
        return

    puzzle = generate_puzzle(seed=args.seed, clues=args.clues, symmetry=args.symmetry)
    print("Puzzle:")
    pretty_print(puzzle)
    print(f"String: {grid_to_string(puzzle)}")

    if args.print_solution:
        sol = solve(puzzle)
        print("Solution:")
        pretty_print(sol)
        print(f"String: {grid_to_string(sol)}")


if __name__ == "__main__":
    main()